# Pneumonia Detection from Chest X-Rays

Transfer learning with a pretrained ResNet18, run cell by cell.

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) from Kaggle.

Make sure the `chest_xray/` folder (with `train/`, `val/`, `test/` subfolders, each containing `NORMAL/` and `PNEUMONIA/`) sits in the same directory as this notebook before running.

## 1. Imports and setup

In [ ]:
import os

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Dataset class

In [ ]:
class PneumoniaDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # These must match your dataset's actual folder names exactly (case-sensitive).
        for label in ['NORMAL', 'PNEUMONIA']:
            class_dir = os.path.join(root_dir, label)
            for img_name in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir, img_name))
                self.labels.append(0 if label == 'NORMAL' else 1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

## 3. Transforms and data loaders

In [ ]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATA_DIR = 'chest_xray'

train_dataset = PneumoniaDataset(root_dir=os.path.join(DATA_DIR, 'train'), transform=data_transform)
val_dataset = PneumoniaDataset(root_dir=os.path.join(DATA_DIR, 'val'), transform=data_transform)
test_dataset = PneumoniaDataset(root_dir=os.path.join(DATA_DIR, 'test'), transform=data_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

## 4. Model: pretrained ResNet18 with a new final layer

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

## 5. Training loop

In [ ]:
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}")

    model.eval()
    val_labels = []
    val_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())

    val_accuracy = accuracy_score(val_labels, val_preds)
    print('Validation Accuracy:', val_accuracy)

## 6. Test set evaluation

In [ ]:
model.eval()
test_labels = []
test_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        test_labels.extend(labels.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())

test_accuracy = accuracy_score(test_labels, test_preds)
print('Test Accuracy:', test_accuracy)
print(classification_report(test_labels, test_preds, target_names=['NORMAL', 'PNEUMONIA']))

## 7. Save the trained model

In [ ]:
torch.save(model.state_dict(), 'pneumonia_classifier.pth')
print("Model saved to pneumonia_classifier.pth")

## 8. Try a prediction on a single image

Run this after training (or after loading a saved model) to test on one new image.

In [ ]:
def predict_single_image(image_path, model):
    class_names = ['NORMAL', 'PNEUMONIA']
    image = Image.open(image_path).convert('RGB')
    input_tensor = data_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        predicted_idx = torch.argmax(probabilities).item()

    print(f"Prediction: {class_names[predicted_idx]}")
    print(f"Confidence: {probabilities[predicted_idx].item() * 100:.2f}%")
    return class_names[predicted_idx], probabilities.cpu().numpy()

# Example usage (update the path to a real image before running):
# predict_single_image('chest_xray/test/PNEUMONIA/some_image.jpeg', model)